In [2]:
pip install nltk


In [3]:
!pip install streamlit
# Import basic libraries
import pandas as pd
import numpy as np
import re
import pickle
import streamlit as st

# NLP libraries
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 86.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.


In [5]:
# Load dataset
df = pd.read_csv("traintoxic.csv")

# Display first 5 rows
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [8]:
toxic_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]
df_new = df.copy()
df_new["toxic_flag"] = df_new[toxic_columns].sum(axis=1)
df_toxic = df_new[df_new["toxic_flag"] > 0]
df_toxic = df_toxic[["comment_text"]]
df_toxic["label"] = "Toxic"
print(df_toxic["label"].value_counts())

label
Toxic    16225
Name: count, dtype: int64


In [9]:
car_negative_samples = [
"The maintenance cost is too high",
"The engine performance is disappointing",
"Fuel consumption is worse than expected",
"The car feels underpowered",
"The gearbox shifts are not smooth",
"The interior quality feels cheap",
"The suspension is too stiff",
"The ride comfort is poor",
"The steering response is slow",
"The car is overpriced for its features",

"The cabin noise is noticeable at high speed",
"The braking performance is not impressive",
"The acceleration is too slow",
"The design looks outdated",
"The infotainment system is confusing",
"The seats are not comfortable for long drives",
"The air conditioning is weak",
"The trunk space is limited",
"The visibility is not very good",
"The headlights are not bright enough",

"The engine makes too much noise",
"The car vibrates at high speeds",
"The build quality could be better",
"The fuel efficiency is disappointing",
"The dashboard design looks basic",
"The car lacks advanced safety features",
"The suspension feels unstable",
"The driving experience is not enjoyable",
"The car struggles on steep roads",
"The materials used inside feel low quality",

"The touchscreen system responds slowly",
"The engine feels weak during overtaking",
"The maintenance schedule is too frequent",
"The fuel tank capacity is too small",
"The car does not feel stable on highways",
"The brakes feel soft",
"The transmission feels jerky",
"The overall performance is below expectations",
"The resale value is not good",
"The warranty coverage is limited",

"The back seat space is cramped",
"The ride quality is not smooth",
"The car feels heavy to drive",
"The turning radius is too wide",
"The engine overheats quickly",
"The paint quality is poor",
"The door closing sound feels cheap",
"The handling is not very precise",
"The service cost is expensive",
"The car lacks power at higher speeds",

"The design is not very attractive",
"The steering wheel feels stiff",
"The rear suspension is noisy",
"The boot space is smaller than expected",
"The fuel mileage is not impressive",
"The engine startup is noisy",
"The overall quality is average",
"The driving comfort is not satisfying",
"The gear transition is rough",
"The car does not justify its price",

"The engine response is delayed",
"The air conditioning takes too long to cool",
"The road noise enters the cabin",
"The materials used feel outdated",
"The seats lack proper support",
"The dashboard rattles while driving",
"The vehicle feels unstable in corners",
"The car struggles with heavy loads",
"The fuel economy is below average",
"The side mirrors are too small",

"The interior design feels old fashioned",
"The acceleration is not responsive",
"The engine lacks refinement",
"The braking distance feels long",
"The rear visibility is limited",
"The infotainment screen is too small",
"The suspension setup is not comfortable",
"The car does not feel premium",
"The gear lever feels loose",
"The engine sound is rough",

"The car feels sluggish in traffic",
"The maintenance cost keeps increasing",
"The performance is inconsistent",
"The cabin space feels tight",
"The steering lacks feedback",
"The overall build quality is weak",
"The car feels unstable at high speed",
"The interior plastics feel hard",
"The suspension absorbs bumps poorly",
"The car is not worth the money"
]
df_car_negative = pd.DataFrame({
    "comment_text": car_negative_samples,
    "label": "Negative"
})

In [10]:
car_positive_samples = [
"This car is amazing and very comfortable",
"The driving experience is smooth and enjoyable",
"The fuel efficiency is excellent",
"I absolutely love this car",
"The engine performance is impressive",
"The interior design looks beautiful",
"The ride quality is very comfortable",
"The steering response is precise",
"The build quality feels premium",
"This car exceeded my expectations",

"The acceleration is smooth and powerful",
"The suspension handles bumps well",
"The cabin is quiet even at high speed",
"The braking system is very reliable",
"The infotainment system works perfectly",
"The seats are extremely comfortable",
"The air conditioning cools quickly",
"The trunk space is spacious",
"The headlights are bright and clear",
"The car feels stable on highways",

"The handling is very responsive",
"The engine is refined and quiet",
"The gearbox shifts smoothly",
"The fuel economy is outstanding",
"The overall performance is excellent",
"The interior materials feel high quality",
"The design looks modern and stylish",
"The ride feels smooth and controlled",
"The steering feels light and accurate",
"The car is very reliable",

"The engine delivers strong performance",
"The driving comfort is impressive",
"The car feels solid and well built",
"The suspension absorbs bumps nicely",
"The acceleration is quick and smooth",
"The braking is sharp and confident",
"The cabin design looks elegant",
"The infotainment display is clear",
"The fuel consumption is very good",
"The overall driving experience is fantastic",

"The seats provide great support",
"The engine response is quick",
"The car feels premium inside",
"The road handling is excellent",
"The vehicle feels stable in corners",
"The interior layout is user friendly",
"The sound system quality is great",
"The design stands out on the road",
"The steering feedback is excellent",
"The car performs well in traffic",

"The gearbox transition is seamless",
"The cabin space is generous",
"The engine runs smoothly",
"The ride comfort is exceptional",
"The car looks very attractive",
"The braking distance is short",
"The performance is consistent",
"The materials used feel durable",
"The car is worth the price",
"The fuel mileage is impressive",

"The suspension setup is well balanced",
"The acceleration is responsive",
"The interior feels luxurious",
"The engine is powerful and smooth",
"The car drives effortlessly",
"The overall quality is excellent",
"The vehicle handles curves confidently",
"The seats are soft and supportive",
"The infotainment system is intuitive",
"The air conditioning works efficiently",

"The steering is very precise",
"The car feels light and agile",
"The design looks sleek and modern",
"The engine is very reliable",
"The cabin feels spacious",
"The ride is smooth and quiet",
"The performance is outstanding",
"The handling feels sharp",
"The car provides great value",
"The overall comfort is excellent",

"The fuel efficiency is better than expected",
"The interior finish is high quality",
"The driving experience is very enjoyable",
"The engine performance is strong",
"The suspension feels comfortable",
"The cabin insulation is excellent",
"The design is very appealing",
"The car feels stable at high speed",
"The steering control is excellent",
"This is one of the best cars I have owned"
]

df_car_positive = pd.DataFrame({
    "comment_text": car_positive_samples,
    "label": "Positive"
})

In [12]:
df_source = pd.read_csv("trainPN.csv") # Load the correct source data
df_sent = df_source.copy()

df_sent = df_sent.rename(columns={
    "review": "comment_text",
    "sentiment": "label"
})

df_sent["label"] = df_sent["label"].map({
    "positive": "Positive",
    "negative": "Negative"
})

df_sent = df_sent[["comment_text", "label"]]
print(df_sent["label"].value_counts())

label
Positive    25000
Negative    25000
Name: count, dtype: int64


In [13]:
df_positive = df_sent[df_sent["label"] == "Positive"].sample(3000, random_state=42)
df_negative = df_sent[df_sent["label"] == "Negative"].sample(3000, random_state=42)
df_toxic = df_toxic.sample(3000, random_state=42)

print("Toxic:", df_toxic.shape)
print("Negative:", df_negative.shape)
print("Positive:", df_positive.shape)

Toxic: (3000, 2)
Negative: (3000, 2)
Positive: (3000, 2)


In [14]:
df_final = pd.concat([df_positive, df_negative, df_toxic, df_car_negative, df_car_positive])
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
print(df_final.shape)
print(df_final["label"].value_counts())

(9180, 2)
label
Positive    3090
Negative    3090
Toxic       3000
Name: count, dtype: int64


In [15]:
nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

df_final["clean_text"] = df_final["comment_text"].apply(clean_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [19]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(7344, 20000)


In [20]:
X = df_final["clean_text"]
y = df_final["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)


(7344, 20000)
(1836, 20000)


In [21]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

print("Naive Bayes")
print(classification_report(y_test, nb_model.predict(X_test_tfidf)))

Naive Bayes
              precision    recall  f1-score   support

    Negative       0.78      0.85      0.81       618
    Positive       0.82      0.83      0.83       618
       Toxic       1.00      0.90      0.95       600

    accuracy                           0.86      1836
   macro avg       0.87      0.86      0.86      1836
weighted avg       0.87      0.86      0.86      1836



In [22]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(class_weight="balanced")
svm_model.fit(X_train_tfidf, y_train)

print("SVM")
print(classification_report(y_test, svm_model.predict(X_test_tfidf)))

SVM
              precision    recall  f1-score   support

    Negative       0.85      0.83      0.84       618
    Positive       0.84      0.86      0.85       618
       Toxic       0.98      0.99      0.98       600

    accuracy                           0.89      1836
   macro avg       0.89      0.89      0.89      1836
weighted avg       0.89      0.89      0.89      1836



In [23]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, class_weight="balanced")
lr_model.fit(X_train_tfidf, y_train)

print("Logistic Regression")
print(classification_report(y_test, lr_model.predict(X_test_tfidf)))

Logistic Regression
              precision    recall  f1-score   support

    Negative       0.87      0.83      0.85       618
    Positive       0.85      0.85      0.85       618
       Toxic       0.96      0.99      0.97       600

    accuracy                           0.89      1836
   macro avg       0.89      0.89      0.89      1836
weighted avg       0.89      0.89      0.89      1836



In [24]:
pickle.dump(svm_model, open("final_model.pkl", "wb"))
pickle.dump(vectorizer, open("tfidf_vectorizer.pkl", "wb"))

In [25]:
def predict_comment(comment):
    cleaned = clean_text(comment)
    vectorized = vectorizer.transform([cleaned])
    prediction = svm_model.predict(vectorized)[0]




    return prediction

In [31]:
test_cases = [
"too enspensive",
"too slow",
"ngfnsgegsdv",
"The fuel economy is outstanding",
"The overall performance is excellent",
"The interior materials feel high quality",
"The design looks good",
"The ride feels smooth and controlled",
"The steering feels light and accurate",
"The car is very reliable",
"The  quality feels cheap"
]

for sentence in test_cases:
    print(sentence, "→", predict_comment(sentence))

too enspensive → Toxic
too slow → Negative
ngfnsgegsdv → Toxic
The fuel economy is outstanding → Positive
The overall performance is excellent → Positive
The interior materials feel high quality → Positive
The design looks good → Positive
The ride feels smooth and controlled → Positive
The steering feels light and accurate → Positive
The car is very reliable → Positive
The  quality feels cheap → Negative


In [32]:
from joblib import dump, load

In [35]:
dump(svm_model, 'svm_model.joblib')

['svm_model.joblib']

In [36]:
import streamlit as st
from joblib import load
import numpy as np

# Load the model
model = load('svm_model.joblib')

st.title("Car Review Comment Classification System")

# User input text
user_input = st.text_area("Enter your car review comment:")

# Predict button
if st.button("Predict Comment Type"):

    if user_input.strip() != "":

        cleaned = clean_text(user_input)
        input_vector = vectorizer.transform([cleaned])
        prediction = model.predict(input_vector)

        st.subheader("Prediction Result:")
        st.success(prediction[0])

    else:
        st.warning("Please enter a comment.")

2026-03-01 07:57:04.661 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 07:57:05.000 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-01 07:57:05.001 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 07:57:05.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 07:57:05.010 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 07:57:05.016 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 07:57:05.023 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 07:57:05.027 Thread 'MainThread': mi